# The _Heliand_

[HeliPaD](https://github.com/DiGS-Corpora/HeliPaD) provides a fully annotated text of the Old Low German _Heliand_, but it does so in the rather arcane CorpusSearch format. Let's see if we can extract the tokens with their line references, part-of-speech tags, and lemmas.

In [69]:
import re
from pathlib import Path
from git import Repo

In [ ]:
# Loop in the HTTPS clone point here:
remote = 'https://github.com/DiGS-Corpora/HeliPaD.git'
# Desired target folder name:
local = '../corpora/HeliPaD'
# Only clone if the target folder doesn't already exist:
if not(Path(local).is_dir()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

In [ ]:
with open('../corpora/HeliPaD/heliand.psd') as infile:
    psd = infile.read().splitlines()

CorpusSearch PSD files contain information on no more than one token per line. Some lines describe constituents of the syntax tree only; these we will ignore for present purposes. There are also lines with syntactical information as well as token markup; these lines use a system of nested parentheses in which the outer parentheses describe the syntactical phrase, and only the innermost pair describes the word itself, using a convention in which the part-of-speech (POS) information is followed by a space, while token form and lemma are separated by a hyphen. So this is a good opportunity to use **regular expressions**. Rather than draw up a list of the many part-of-speech tags used for annotation, we will single out sequences 

1. beginning in an opening parenthesis (we will have to escape these with a backslash), 
2. followed by a string of characters that will be our **POS tag**, 
3. followed by a space, 
4. followed by a sequence of letters that will be our **word form**, 
5. followed by a hyphen, 
6. followed by a sequence of letters that will be our **lemma**, 
7. followed by a closing parenthesis. Easy!

Regular expressions even allow us to single out **match groups** within our pattern (indicated by non-escaped parentheses), so the return contains separate containers for the three kinds of information we are after.

In [ ]:

pattern = re.compile(r"\(([A-Z0-9^+=*$-]*)\s(\w*)-([^)]*)\)")

The final two pieces of information we may want to store are line number and halfline, so we can reconstruct verse lines downstream if we want.

In [77]:
line_boundary = re.compile(r"\(CODE <R_(\d*)")
caesura = re.compile(r"\(CODE <C>")

In [78]:
tokens = []
line_num = 1
halfline = 'a'
for line in psd:
    token = dict()
    token['line'] = str(line_num) + halfline
    result = pattern.search(line)
    newline = line_boundary.search(line)
    off_verse = caesura.search(line)
    if result:
        token['form'] = result.group(2)
        token['lemma'] = result.group(3)
        token['pos'] = result.group(1)
        tokens.append(token)
    elif off_verse:
        halfline = 'b'
    elif newline:
        line_num = int(newline.group(1))
        halfline = 'a'

In [79]:
tokens[205]

{'line': '30a', 'form': 'mildean', 'lemma': 'mildi', 'pos': 'ADJ^A^SG'}

Now we can also isolate any one kind of information. For instance, we can write our tokens to file one (half)line at a time, reconstruction the edition (Sievers's) on which the PSD file was based:

In [88]:
verse_lines = []
for number in range(1, int(tokens[-1]['line'].rstrip('ab'))):
    hits_a = [i['form'] for i in tokens if i['line'] == str(number) + 'a']
    hits_b = [i['form'] for i in tokens if i['line'] == str(number) + 'b']
    reconstructed_line = ' '.join(hits_a) + '    ' + ' '.join(hits_b)
    verse_lines.append(reconstructed_line)

In [87]:
verse_lines[0]

'Manega uuaron    the sia iro mod gespon'

Finally, we'll write the reconstructed lines to disk:

In [89]:
Path('../corpora/heliand').mkdir(parents=True, exist_ok=True)
with open('../corpora/heliand/heliand_c.txt', 'w') as outfile:
    outfile.write('\n'.join(verse_lines))

The C text of the _Heliand_ lacks the last fifteen lines of M.